# Recurrent Neural Networks from Scratch

**Learning objectives**
- See why feedforward nets struggle with variable-length sequences
- Implement a vanilla RNN cell and train it with BPTT in NumPy
- Predict the next character in a tiny repeating language
- Compare with PyTorch `nn.RNN`

Run cells top-to-bottom. Constants are grouped near the top so you can experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Hyperparameters (tweak these) ---
HIDDEN_SIZE = 32
SEQ_LEN = 12
LEARNING_RATE = 0.05
EPOCHS = 200
BATCH_SIZE = 32
N_TRAIN = 512
N_TEST = 128

## 1. Problem setup — next-character prediction

A **language** of three tokens `{a, b, c}` with a simple grammar:

> After `a` always comes `b`; after `b` always comes `c`; after `c` always comes `a`.

So the infinite string is `abcabcabc...`. The model sees a window and must predict the **next token**.

This is trivial for humans, but it forces the network to use **recurrent state** rather than a bag of characters.

In [ ]:
VOCAB = ['a', 'b', 'c']
stoi = {ch: i for i, ch in enumerate(VOCAB)}
itos = {i: ch for ch, i in stoi.items()}
V = len(VOCAB)


def cycle_char(i):
    return VOCAB[i % 3]


def make_sequences(n, seq_len=SEQ_LEN, rng=None):
    # X: (n, T) token ids, Y: (n,) next-token id after the window.
    rng = rng or np.random.default_rng(0)
    starts = rng.integers(0, 3, size=n)
    X = np.zeros((n, seq_len), dtype=np.int64)
    y = np.zeros(n, dtype=np.int64)
    for i, s in enumerate(starts):
        for t in range(seq_len):
            X[i, t] = stoi[cycle_char(s + t)]
        y[i] = stoi[cycle_char(s + seq_len)]
    return X, y


X_train, y_train = make_sequences(N_TRAIN, rng=np.random.default_rng(0))
X_test, y_test = make_sequences(N_TEST, rng=np.random.default_rng(1))

print('example window → next')
for i in range(5):
    window = ''.join(itos[t] for t in X_train[i])
    print(f'  {window}  →  {itos[y_train[i]]}')

## 2. The RNN recurrence

At each time $t$, hidden state $h_t$ mixes the new input $x_t$ with the previous state $h_{t-1}$:

$$
h_t = \tanh\!\big(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b_h\big)
$$

$$
\hat{y} = \mathrm{softmax}\!\big(W_{hy}\, h_T + b_y\big)
$$

(We classify from the **final** hidden state $h_T$ — many-to-one.)

**Learning note:** $W_{hh}$ is reused every step. That is what lets an RNN process length-$T$ sequences with a parameter count independent of $T$ — and also what makes gradients multiply $T$ times (vanishing/exploding).

In [ ]:
def one_hot_seq(X, v=V):
    # X (N,T) -> (N,T,V)
    N, T = X.shape
    out = np.zeros((N, T, v))
    for n in range(N):
        out[n, np.arange(T), X[n]] = 1.0
    return out


def init_rnn(h=HIDDEN_SIZE, v=V, rng=np.random.default_rng(42)):
    scale = 0.1
    return {
        'Wxh': rng.normal(0, scale, size=(v, h)),
        'Whh': rng.normal(0, scale, size=(h, h)),
        'bh': np.zeros((1, h)),
        'Why': rng.normal(0, scale, size=(h, v)),
        'by': np.zeros((1, v)),
    }


def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)


def rnn_forward(X_ids, params):
    # Many-to-one: return probs (N,V) and cache for BPTT.
    X = one_hot_seq(X_ids)  # (N,T,V)
    N, T, _ = X.shape
    h = HIDDEN_SIZE
    hs = [np.zeros((N, h))]  # h_0
    pretanh = []
    for t in range(T):
        z = X[:, t] @ params['Wxh'] + hs[-1] @ params['Whh'] + params['bh']
        ht = np.tanh(z)
        pretanh.append(z)
        hs.append(ht)
    logits = hs[-1] @ params['Why'] + params['by']
    probs = softmax(logits)
    cache = (X, hs, pretanh, probs)
    return probs, cache


params = init_rnn()
probs, cache = rnn_forward(X_train[:3], params)
print('init probs:\n', np.round(probs, 3))

## 3. Backpropagation Through Time (BPTT)

Loss is categorical CE on the final prediction. Let $\delta_y = \hat{y} - y$.

Gradients for the output layer are standard. For the hidden state, we recurse **backward in time**:

$$
\delta_t = \big(\delta_{t+1}\, W_{hh}^\top + \mathbf{1}_{t=T}\,\delta_y\, W_{hy}^\top\big) \odot \tanh'(z_t)
$$

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} h_{t-1}^\top \delta_t, \qquad
\frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^{T} x_t^\top \delta_t
$$

**Learning note:** Each step multiplies by $W_{hh}^\top$ and by $\tanh' \le 1$. Over long $T$, products of small numbers → **vanishing gradients** (see the LSTM notebook).

In [ ]:
def one_hot_y(y, v=V):
    Y = np.zeros((len(y), v))
    Y[np.arange(len(y)), y] = 1.0
    return Y


def rnn_backward(y, params, cache):
    X, hs, pretanh, probs = cache
    N, T, V_ = X.shape
    Y = one_hot_y(y)
    dlogits = (probs - Y) / N

    grads = {k: np.zeros_like(v) for k, v in params.items()}
    grads['Why'] = hs[-1].T @ dlogits
    grads['by'] = dlogits.sum(axis=0, keepdims=True)

    dh_next = dlogits @ params['Why'].T  # dL/dh_T
    for t in reversed(range(T)):
        # dh/dz = 1 - tanh^2
        dz = dh_next * (1.0 - np.tanh(pretanh[t]) ** 2)
        grads['Wxh'] += X[:, t].T @ dz
        grads['Whh'] += hs[t].T @ dz          # hs[t] is h_{t-1}
        grads['bh'] += dz.sum(axis=0, keepdims=True)
        dh_next = dz @ params['Whh'].T         # pass to previous time
    return grads


def clip_grads(grads, max_norm=5.0):
    total = np.sqrt(sum(np.sum(g ** 2) for g in grads.values()))
    if total > max_norm:
        scale = max_norm / (total + 1e-8)
        for k in grads:
            grads[k] *= scale
    return grads


def loss_acc(probs, y):
    Y = one_hot_y(y)
    eps = 1e-8
    loss = float(-np.mean(np.sum(Y * np.log(np.clip(probs, eps, 1)), axis=1)))
    acc = float(np.mean(np.argmax(probs, axis=1) == y))
    return loss, acc


g = rnn_backward(y_train[:3], params, cache)
print('grad norms:', {k: float(np.linalg.norm(v)) for k, v in g.items()})

### Pause & Reflect — BPTT

1. Why do we sum $h_{t-1}^\top \delta_t$ over all $t$ for $\partial L/\partial W_{hh}$?
2. What does gradient clipping protect against?
3. If $\tanh'(z_t)\approx 0$ for many $t$, what happens to early-time learning?

### Discussion — BPTT

1. **Shared weights**: $W_{hh}$ is the same matrix at every step, so its total gradient is the sum of per-step contributions (like tied embeddings).
2. **Clipping**: Stops a single large $\prod W_{hh}$ product from producing NaNs / exploding updates.
3. **Saturation**: Near-zero $\tanh'$ blocks the chain rule — early tokens get almost no credit assignment (vanishing).

## 4. Training

In [ ]:
def train_rnn(X, y, X_te, y_te, epochs=EPOCHS, lr=LEARNING_RATE, batch=BATCH_SIZE):
    params = init_rnn()
    rng = np.random.default_rng(0)
    hist = {'loss': [], 'test_acc': []}
    n = len(y)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        for start in range(0, n, batch):
            bi = idx[start:start + batch]
            probs, cache = rnn_forward(X[bi], params)
            grads = clip_grads(rnn_backward(y[bi], params, cache))
            for k in params:
                params[k] -= lr * grads[k]
        tr_p, _ = rnn_forward(X, params)
        te_p, _ = rnn_forward(X_te, params)
        tr_loss, _ = loss_acc(tr_p, y)
        _, te_acc = loss_acc(te_p, y_te)
        hist['loss'].append(tr_loss)
        hist['test_acc'].append(te_acc)
        if (epoch + 1) % 40 == 0 or epoch == 0:
            print(f'epoch {epoch+1:3d}  loss={tr_loss:.3f}  test={te_acc*100:.1f}%')
    return params, hist


params, hist = train_rnn(X_train, y_train, X_test, y_test)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(hist['loss'], color='steelblue')
ax2.plot([a * 100 for a in hist['test_acc']], color='coral')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax2.set_ylabel('test acc %')
ax1.set_title('RNN next-token training'); ax1.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Generate by rolling the RNN forward one step at a time
def generate(params, seed='ab', n=30):
    ids = [stoi[c] for c in seed]
    for _ in range(n):
        window = np.array([ids[-SEQ_LEN:]], dtype=np.int64)
        if window.shape[1] < SEQ_LEN:
            # left-pad with zeros (token 'a') for short seeds — demo only
            pad = np.zeros((1, SEQ_LEN - window.shape[1]), dtype=np.int64)
            window = np.concatenate([pad, window], axis=1)
        probs, _ = rnn_forward(window, params)
        ids.append(int(np.argmax(probs[0])))
    return ''.join(itos[i] for i in ids)


print('generated:', generate(params, seed='abcabcabcabc'))
print('expected :', ('abc' * 20)[:42])

## 5. PyTorch comparison

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

class CharRNN(nn.Module):
    def __init__(self, v=V, h=HIDDEN_SIZE):
        super().__init__()
        self.embed = nn.Embedding(v, h)  # learned embedding ≈ one-hot @ Wxh
        self.rnn = nn.RNN(h, h, batch_first=True)
        self.fc = nn.Linear(h, v)
    def forward(self, x):
        # x: (N,T) long
        out, h_n = self.rnn(self.embed(x))
        return self.fc(h_n.squeeze(0))  # last hidden


model = CharRNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
crit = nn.CrossEntropyLoss()
loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=BATCH_SIZE, shuffle=True,
)
X_te_t = torch.tensor(X_test)
y_te_t = torch.tensor(y_test)

torch_accs = []
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t).argmax(1)
        torch_accs.append((pred == y_te_t).float().mean().item())
    if (epoch + 1) % 40 == 0 or epoch == 0:
        print(f'PyTorch epoch {epoch+1:3d}  test={torch_accs[-1]*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([a * 100 for a in hist['test_acc']], label='NumPy RNN', linewidth=2)
ax.plot([a * 100 for a in torch_accs], label='PyTorch RNN', linewidth=2)
ax.set_xlabel('epoch'); ax.set_ylabel('test accuracy %')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_title('NumPy vs PyTorch RNN')
plt.tight_layout(); plt.show()

## Summary

| Concept | Meaning |
|---------|---------|
| Hidden state $h_t$ | Compressed memory of the prefix $x_{1:t}$ |
| Shared $W_{hh}$ | Parameter count ≠ sequence length |
| BPTT | Unroll the loop, backprop through the chain |
| Failure mode | Vanishing/exploding grads on long dependencies |

**Next:** LSTMs add gates so memory can persist across many steps.